# Лабораторная работа 3: PySpark
## Большие данные и рекомендательные системы на MovieLens 1M
**Студент:** Шибитов Николай  
**Группа:** 11  
**Курс:** 3  
**Дисциплина:** Искусственный интеллект  
**Дата:** 2026-05-13


## Задание -1: Теория

### Что такое PySpark и в чём его главное отличие от Pandas?

**PySpark** — это Python-интерфейс для Apache Spark, фреймворка для распределённой обработки данных. Он позволяет писать код на Python, но выполнять вычисления средствами Spark, то есть не только на одной машине, а также на нескольких узлах кластера.

Главное отличие от **Pandas** состоит в масштабе и способе выполнения. Pandas обычно работает с данными в оперативной памяти одной машины. Если датасет становится слишком большим, работа может замедлиться или вообще не выполниться из-за нехватки памяти. PySpark рассчитан на большие данные: он может разбивать вычисления на части и выполнять их параллельно.

При этом Pandas удобнее для небольших таблиц и быстрого анализа, а PySpark нужен тогда, когда данных много или требуется распределённая обработка.

### Что такое PySpark MLlib и в чём её главное отличие от Sklearn?

**PySpark MLlib** — это библиотека машинного обучения внутри Spark. Она позволяет обучать модели на больших датасетах, используя распределённые вычисления.

Главное отличие от **Scikit-learn** заключается в том, что Sklearn в основном работает на одной машине, а MLlib может выполнять обучение на кластере. Sklearn обычно удобнее и богаче по количеству алгоритмов для классических задач на небольших данных. MLlib полезнее, когда данные уже находятся в Spark DataFrame и имеют большой объём.

### В чём концепция Lazy Evaluation в PySpark?

**Ленивые вычисления** означают, что Spark не выполняет трансформации сразу после их вызова. Например, если написать `filter`, `select` или `groupBy`, Spark только запоминает последовательность операций и строит логический план вычислений.

Реальное выполнение начинается только тогда, когда вызывается действие, например `show()`, `count()` или `collect()`. Благодаря этому Spark может оптимизировать план: убрать лишние операции, изменить порядок выполнения и выполнить вычисления эффективнее.

### В чём разница между Transformations и Actions?

**Transformations** — это операции, которые создают новый DataFrame или RDD, но сразу не запускают вычисления. Примеры: `select`, `filter`, `withColumn`, `groupBy`, `join`.

**Actions** — это операции, которые требуют получить результат и поэтому запускают выполнение накопленного плана. Примеры: `show`, `count`, `collect`, `take`, `write`.

### Что такое RDD, DataFrame и SparkSession?

**RDD** — это базовая низкоуровневая распределённая структура данных Spark. Она даёт контроль над операциями, но менее удобна для аналитики.

**DataFrame** — это таблица с именованными столбцами, похожая на таблицу в SQL или Pandas. DataFrame удобнее для анализа, а Spark умеет оптимизировать операции над ним.

**SparkSession** — это главная точка входа в Spark-приложение. Через неё создаются DataFrame, выполняются SQL-запросы и настраивается работа Spark.

### Как устроена базовая архитектура Spark-приложения?

Spark-приложение состоит из **Driver** и **Executors**. Driver запускает основную программу, строит план выполнения и распределяет задачи. Executors выполняют эти задачи, хранят промежуточные данные и возвращают результаты Driver'у.

Если программа запускается в режиме `local[*]`, то Spark работает локально, но всё равно использует ту же модель: Driver управляет задачами, а вычисления выполняются параллельно в доступных потоках.

### Что такое Apache Parquet и зачем он нужен?

**Apache Parquet** — это колоночный формат хранения данных. В отличие от обычного CSV, Parquet хранит данные по столбцам, поддерживает сжатие и сохраняет схему данных.

Он удобен для аналитики, потому что Spark может читать только нужные столбцы, а не весь файл целиком. Поэтому Parquet обычно быстрее и компактнее CSV, особенно при работе с большими датасетами.

---
## Задание 0: Подготовка

Перед запуском ноутбука необходимо установить PySpark. В задании указано `pip install spark`, но для работы с модулем `pyspark` обычно используется установка пакета `pyspark`.

In [ ]:
# Если PySpark не установлен, раскомментируйте строку ниже и выполните ячейку:
# !pip install pyspark

In [ ]:
import os
import zipfile

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, split, explode, lit, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
spark = (SparkSession.builder
    .appName("MovieLens Lab")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

---
## Задание 1: Предобработка

Используется датасет **MovieLens 1M**. В работе нужны файлы `movies.dat` и `ratings.dat`. Если в папке с ноутбуком находится архив `ml-1m.zip`, следующая ячейка автоматически распакует его.

In [ ]:
# Автоматическая распаковка архива, если он лежит рядом с ноутбуком
zip_path = "ml-1m.zip"
extract_dir = "."

if os.path.exists(zip_path) and not os.path.exists("ml-1m"):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Архив ml-1m.zip распакован")
else:
    print("Распаковка не требуется или архив не найден")

In [ ]:
# Пути к файлам MovieLens 1M
movies_path = "ml-1m/movies.dat"
ratings_path = "ml-1m/ratings.dat"

print("movies.dat найден:", os.path.exists(movies_path))
print("ratings.dat найден:", os.path.exists(ratings_path))

In [ ]:
# Схемы данных задаются явно, так как в файлах нет заголовков
movies_schema = StructType([
    StructField("movieID", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userID", IntegerType(), True),
    StructField("movieID", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", IntegerType(), True)
])

In [ ]:
# Загрузка данных. Файлы MovieLens используют разделитель :: и кодировку ISO-8859-1.
movies_df = (spark.read
    .option("sep", "::")
    .option("encoding", "ISO-8859-1")
    .schema(movies_schema)
    .csv(movies_path)
)

ratings_df = (spark.read
    .option("sep", "::")
    .option("encoding", "ISO-8859-1")
    .schema(ratings_schema)
    .csv(ratings_path)
)

print("Количество фильмов:", movies_df.count())
print("Количество оценок:", ratings_df.count())

movies_df.show(5, truncate=False)
ratings_df.show(5)

In [ ]:
# Кэшируем ratings_df, потому что далее он используется много раз:
# в аналитике, join, SQL-запросах, обучении ALS и рекомендациях.
ratings_df.cache()

# Действие count() материализует кэш, то есть реально загружает данные в память Spark.
ratings_count = ratings_df.count()
print("ratings_df закэширован. Количество строк:", ratings_count)

**Вывод:** Кэширование полезно, когда один и тот же DataFrame используется повторно. В этой лабораторной `ratings_df` нужен в нескольких заданиях, поэтому Spark может не перечитывать и не пересчитывать его каждый раз, а брать данные из кэша.

In [ ]:
# Разделяем строку жанров на массив и «взрываем» массив так,
# чтобы каждый жанр фильма оказался в отдельной строке.
movies_genres_df = movies_df.withColumn("genre", explode(split(col("genres"), "\\|")))

movies_genres_df.show(10, truncate=False)

---
## Задание 2: Аналитика

In [ ]:
# 2.1 Топ-10 самых популярных жанров по количеству фильмов
popular_genres = (movies_genres_df
    .groupBy("genre")
    .agg(count("movieID").alias("movies_count"))
    .orderBy(col("movies_count").desc())
    .limit(10)
)

popular_genres.show(truncate=False)

In [ ]:
# 2.2 Топ-10 самых активных пользователей по количеству выставленных оценок
active_users = (ratings_df
    .groupBy("userID")
    .agg(count("rating").alias("ratings_count"), avg("rating").alias("avg_rating"))
    .orderBy(col("ratings_count").desc())
    .limit(10)
)

active_users.show(truncate=False)

**Вывод:** Популярность жанра здесь считается по количеству фильмов данного жанра в каталоге. Дополнительно были найдены самые активные пользователи — те, кто поставил больше всего оценок.

---
## Задание 3: Объединение данных (Join)

In [ ]:
# 3.1 Топ-10 фильмов по количеству оценок
popular_movies = (ratings_df
    .groupBy("movieID")
    .agg(count("rating").alias("ratings_count"))
    .orderBy(col("ratings_count").desc())
    .limit(10)
    .join(movies_df, on="movieID", how="inner")
    .select("movieID", "title", "genres", "ratings_count")
)

popular_movies.show(truncate=False)

In [ ]:
# 3.2 Топ-10 лучших фильмов по среднему рейтингу среди фильмов, у которых не менее 500 оценок
best_movies = (ratings_df
    .groupBy("movieID")
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("ratings_count")
    )
    .filter(col("ratings_count") >= 500)
    .orderBy(col("avg_rating").desc(), col("ratings_count").desc())
    .limit(10)
    .join(movies_df, on="movieID", how="inner")
    .select("movieID", "title", "genres", "avg_rating", "ratings_count")
)

best_movies.show(truncate=False)

### Демонстрация видов JOIN

Для демонстрации создаются маленькие таблицы: в первой есть `id = 1, 2`, во второй — `id = 2, 3`. По результатам хорошо видно, какие строки сохраняет каждый тип соединения.

In [ ]:
left_df = spark.createDataFrame([
    (1, "A"),
    (2, "B")
], ["id", "left_value"])

right_df = spark.createDataFrame([
    (2, "X"),
    (3, "Y")
], ["id", "right_value"])

print("Левая таблица")
left_df.show()

print("Правая таблица")
right_df.show()

In [ ]:
print("INNER JOIN: остаются только совпавшие id")
left_df.join(right_df, on="id", how="inner").show()

print("LEFT JOIN: остаются все строки из левой таблицы")
left_df.join(right_df, on="id", how="left").show()

print("RIGHT JOIN: остаются все строки из правой таблицы")
left_df.join(right_df, on="id", how="right").show()

print("OUTER JOIN: остаются все строки из обеих таблиц")
left_df.join(right_df, on="id", how="outer").show()

print("CROSS JOIN: декартово произведение всех строк")
left_df.crossJoin(right_df).show()

**Вывод:** `inner` оставляет только совпадения, `left` сохраняет все строки слева, `right` — все строки справа, `outer` объединяет все строки из обеих таблиц, а `cross` создаёт все возможные пары строк.

---
## Задание 4: Spark SQL

In [ ]:
# Создаём временные представления, чтобы обращаться к DataFrame через SQL
ratings_df.createOrReplaceTempView("ratings")
movies_df.createOrReplaceTempView("movies")

In [ ]:
# Топ-10 фильмов с >= 500 оценок через spark.sql()
best_movies_sql = spark.sql('''
SELECT
    m.movieID,
    m.title,
    m.genres,
    AVG(r.rating) AS avg_rating,
    COUNT(r.rating) AS ratings_count
FROM ratings r
INNER JOIN movies m
    ON r.movieID = m.movieID
GROUP BY m.movieID, m.title, m.genres
HAVING COUNT(r.rating) >= 500
ORDER BY avg_rating DESC, ratings_count DESC
LIMIT 10
''')

best_movies_sql.show(truncate=False)

**Вывод:** Результат SQL-запроса должен совпадать по смыслу с результатом из задания 3. Разница только в способе записи: в задании 3 использовался DataFrame API, а здесь — SQL.

---
## Задание 5: Spark MLlib

Для построения рекомендаций используется алгоритм **ALS** — Alternating Least Squares. Он подходит для рекомендательных систем, потому что пытается найти скрытые факторы пользователей и фильмов на основе матрицы оценок.

In [ ]:
# Делим данные на обучающую и тестовую выборки
train_df, test_df = ratings_df.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count())
print("Test:", test_df.count())

In [ ]:
als = ALS(
    maxIter=5,
    regParam=0.01,
    userCol="userID",
    itemCol="movieID",
    ratingCol="rating",
    coldStartStrategy="drop"
)

model = als.fit(train_df)
print("Модель ALS обучена")

In [ ]:
# Получаем предсказания на тестовой выборке
predictions = model.transform(test_df)

predictions.select("userID", "movieID", "rating", "prediction").show(10)

In [ ]:
# Оцениваем качество по RMSE
# Чем меньше RMSE, тем ближе предсказания модели к реальным оценкам.
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse:.4f}")

**Вывод:** RMSE показывает среднюю ошибку прогноза в шкале рейтингов. Например, если RMSE около 0.9, это означает, что предсказания модели в среднем ошибаются примерно меньше чем на один балл рейтинга.

---
## Задание 6: Персонализация рекомендаций

Выберем одного пользователя и получим для него 10 рекомендованных фильмов методом `recommendForUserSubset(..., 10)`.

In [ ]:
# Можно поменять user_id на любой существующий id пользователя
user_id = 1

user_subset = spark.createDataFrame([(user_id,)], ["userID"])

user_recommendations = model.recommendForUserSubset(user_subset, 10)
user_recommendations.show(truncate=False)

In [ ]:
# Для читаемого вывода развернём массив рекомендаций и присоединим названия фильмов
recommended_movies = (user_recommendations
    .select("userID", explode("recommendations").alias("rec"))
    .select(
        "userID",
        col("rec.movieID").alias("movieID"),
        col("rec.rating").alias("predicted_rating")
    )
    .join(movies_df, on="movieID", how="left")
    .select("userID", "movieID", "title", "genres", "predicted_rating")
    .orderBy(col("predicted_rating").desc())
)

recommended_movies.show(10, truncate=False)

**Вывод:** Для выбранного пользователя модель сформировала персональный список фильмов. Значение `predicted_rating` — это не реальная оценка, а прогноз модели ALS, насколько пользователю может понравиться фильм.

---
## Итоговый вывод

В лабораторной работе был загружен датасет MovieLens 1M, выполнена предобработка данных, проведена аналитика по жанрам, фильмам и пользователям, показана работа разных видов `join`, выполнен SQL-запрос через Spark SQL, а также построена рекомендательная модель ALS из PySpark MLlib. В конце были получены персональные рекомендации для выбранного пользователя.

---
## Завершение Spark-сессии

В конце работы Spark-сессию нужно остановить. Это освобождает память, кэш и фоновые процессы.


In [ ]:
# Завершаем Spark-сессию, чтобы освободить память, кэш и фоновые процессы.
spark.stop()
print("SparkSession остановлена")
